# Phase 2 walkthrough: the rule engine

This notebook runs Phase 2 and then walks through it. Section 0 notes the committed results and re-runs the rules-only baseline, and section 7 checks that the re-run reproduces them. The test set is never opened, and the machine-learning outputs of the next phase are not read here.

**How to use it:** run the cells in order. The **Try it** cell is for your own values.

In [1]:
import os
import re
import subprocess
import sys
import time
from pathlib import Path

ROOT = Path.cwd().parent
sys.path.insert(0, str(ROOT / "src"))

import numpy as np
import pandas as pd
from IPython.display import Image, display

pd.set_option("display.width", 140)
pd.set_option("display.max_columns", 20)


def run_step(script, *args, show=25):
    """Run one project script from src/ with this notebook's Python, from the repository root."""
    t0 = time.perf_counter()
    env = {**os.environ, "PYTHONIOENCODING": "utf-8"}
    r = subprocess.run([sys.executable, "-X", "faulthandler=0", str(ROOT / "src" / script), *args], cwd=ROOT,
                       capture_output=True, text=True, encoding="utf-8", errors="replace", env=env)
    lines = (r.stdout or "").rstrip().splitlines()
    if show:
        if len(lines) > show:
            print(f"... ({len(lines) - show} earlier lines not shown)")
        print("\n".join(lines[-show:]))
    print(f"[{script} {' '.join(args)} finished in {time.perf_counter() - t0:.1f} s, exit code {r.returncode}]")
    if r.returncode != 0:
        print(r.stderr[-3000:])
        raise RuntimeError(f"{script} failed")


print("Research prototype, not a diagnostic tool. The test set is not used in this notebook.")

from data import CLASS_ORDER, TARGET, load_dev
from rules import SOURCES, VARIANTS, Thresholds, run

Research prototype, not a diagnostic tool. The test set is not used in this notebook.


## 0. Run Phase 2

We note the committed rules-only report (a snapshot), then re-run `src/phase2_rules_baseline.py`. It scores the rule engine on the 963 clean development rows and freezes the label-cascade baseline.

In [2]:
P2 = ROOT / "results" / "phase2"
before = (P2 / "rules_baseline.txt").read_text(encoding="utf-8")
before_cascade = (P2 / "cascade_frozen.json").read_text(encoding="utf-8")
run_step("phase2_rules_baseline.py", show=12)

... (66 earlier lines not shown)
    Leukemia: 25/35, size 1.00, misses {'X9': 10}
    Macrocytic anemia: 11/13, size 1.31, misses {'X2': 2}
    Leukemia with thrombocytopenia: 3/9, size 1.67, misses {'X9': 6}

== C. rows between a cited cut-off and the label-cascade cut-off (Phase 1, section G) ==
  WBC in (10, 11] (cascade says high, cited rule says normal): 35 rows, by label {'Leukemia': 10, 'Normocytic hypochromic anemia': 8, 'Iron deficiency anemia': 8, 'Leukemia with thrombocytopenia': 6, 'Normocytic normochromic anemia': 3}
  normocytic anemia with MCHC in [30, 32) (cascade says hypochromic, cited rule says normochromic): 145 rows, by label {'Normocytic hypochromic anemia': 145}

== D. label-cascade baseline (option B), frozen in cascade_frozen.json ==
  fitted MCHC split (iron deficiency below, other microcytic at or above): 31.9
  IN-SAMPLE development agreement (optimistic, the cut-offs were read off these rows): 946 / 963 (98.2%)
  in-sample balanced accuracy over 9 classes:

## 1. The thresholds the code actually uses

Each cut-off was cited and logged on 14 Nov 2026, before any machine-learning result existed. The table is built from the code itself, so it cannot drift from what the engine does.

In [3]:
t = Thresholds()
rows = [("anemia if HGB below", t.hgb_anemia, "g/dL", "WHO"), ("microcytic if MCV below", t.mcv_micro, "fL", "MERCK_ANEMIA"),
        ("macrocytic if MCV above", t.mcv_macro, "fL", "MERCK_ANEMIA"), ("hypochromic if MCHC below", t.mchc_hypo, "g/dL", "MERCK_ANEMIA"),
        ("low platelets if PLT below", t.plt_low, "10^9/L", "AAFP_PLT"), ("leukocytosis if WBC above", t.wbc_high, "10^9/L", "AAFP_WBC"),
        ("leukopenia if WBC below", t.wbc_low, "10^9/L", "MERCK_WBC")]
pd.DataFrame([(a, b, c, SOURCES[d]) for a, b, c, d in rows], columns=["rule", "cut-off", "unit", "source"])

,rule,cut-off,unit,source
0,anemia if HGB below,13.0,g/dL,"WHO 2024, Guideline on haemoglobin cutoffs to ..."
1,microcytic if MCV below,80.0,fL,"Gerber, Evaluation of Anemia, Merck Manual Pro..."
2,macrocytic if MCV above,100.0,fL,"Gerber, Evaluation of Anemia, Merck Manual Pro..."
3,hypochromic if MCHC below,30.0,g/dL,"Gerber, Evaluation of Anemia, Merck Manual Pro..."
4,low platelets if PLT below,150.0,10^9/L,"Gauer and Whitaker, Am Fam Physician 2022;106(..."
5,leukocytosis if WBC above,11.0,10^9/L,"Riley and Rupert, Am Fam Physician 2015;92(11)..."
6,leukopenia if WBC below,4.0,10^9/L,"Dale, Overview of Leukopenias, Merck Manual Pr..."


## 2. Forward chaining, one step at a time

Working memory starts with the CBC values. Each rule fires when the facts it needs exist, and its result becomes a new fact. Later rules build on earlier ones (F9 uses F7 and F8). It stops when no rule can add anything. Below is the made-up record from the unit tests.

In [4]:
example = {"WBC": 7.4, "RBC": 5.0, "HGB": 10.0, "MCV": 70.0, "MCH": 22.0, "MCHC": 29.0, "PLT": 120.0}
res = run(example)
for step in res.trace:
    print(f"{step['rule']:>4}  {step['text']}")
    print(f"      inputs {step['inputs']} -> {step['result']}")
print("\ncompatible set:", res.compatible)
print("excluded (label: rules):", res.excluded)

  F1  HGB < 13 g/dL -> anemia
      inputs {'HGB': 10.0} -> {'anemia': True}
  F2  MCV < 80 fL -> microcytic
      inputs {'MCV': 70.0} -> {'microcytic': True}
  F3  MCV > 100 fL -> macrocytic
      inputs {'MCV': 70.0} -> {'macrocytic': False}
  F4  neither microcytic nor macrocytic -> normocytic
      inputs {} -> {'normocytic': False}
  F5  MCHC < 30 g/dL -> hypochromic
      inputs {'MCHC': 29.0} -> {'hypochromic': True}
  F6  PLT < 150 x10^9/L -> low platelets
      inputs {'PLT': 120.0} -> {'low_plt': True}
  F7  WBC > 11 x10^9/L -> leukocytosis
      inputs {'WBC': 7.4} -> {'leukocytosis': False}
  F8  WBC < 4 x10^9/L -> leukopenia
      inputs {'WBC': 7.4} -> {'leukopenia': False}
  F9  leukocytosis or leukopenia -> abnormal WBC
      inputs {} -> {'abnormal_wbc': False}
 F10  low platelets or abnormal WBC -> provisional rule-based REFER trigger
      inputs {} -> {'refer_rule': True}
  I1  microcytic anemia -> Mentzer MCV/RBC (< 13 favours thalassemia trait)
      inputs {'MCV

## 3. Exclusion only, on three contrasting records

Rules only remove labels that a cited finding rules out. They never name the answer. Look at what stays in each set and why.

In [5]:
NORMAL = {"WBC": 7.0, "RBC": 4.8, "HGB": 14.0, "MCV": 88.0, "MCH": 29.0, "MCHC": 33.0, "PLT": 250.0}
cases = {"anemia + low platelets": {**NORMAL, "HGB": 10, "MCV": 70, "PLT": 100},
         "normocytic anemia, MCHC 31": {**NORMAL, "HGB": 11, "MCHC": 31},
         "WBC 10.5, otherwise normal": {**NORMAL, "WBC": 10.5}}
for name, rec in cases.items():
    print(f"{name:28s} primary: {run(rec).compatible}")
    print(f"{'':28s} MCHC 32 check: {run(rec, VARIANTS['mchc32']).compatible}")

anemia + low platelets       primary: ['Iron deficiency anemia', 'Thrombocytopenia', 'Other microcytic anemia']
                             MCHC 32 check: ['Iron deficiency anemia', 'Thrombocytopenia', 'Other microcytic anemia']
normocytic anemia, MCHC 31   primary: ['Normocytic normochromic anemia']
                             MCHC 32 check: ['Normocytic hypochromic anemia']
WBC 10.5, otherwise normal   primary: ['Healthy']
                             MCHC 32 check: ['Healthy']


The second case shows the MCHC gap: with the cited cut-off (30) an MCHC of 31 counts as normochromic, while with 32 it counts as hypochromic. The third case shows the WBC gap: 10.5 is normal under the cited cut-off (11), but the labels seem to treat anything above 10 as high.

**Try it:** change the values and run the cell.

In [6]:
mine = {**NORMAL, "HGB": 12.5, "MCV": 102, "PLT": 140}
r = run(mine)
print("primary:", r.compatible, "| refer trigger:", r.refer_rule_fires)
print("HGB 12 check:", run(mine, VARIANTS["hgb12"]).compatible)

primary: ['Thrombocytopenia', 'Macrocytic anemia'] | refer trigger: True
HGB 12 check: ['Thrombocytopenia']


## 4. Rules-only baseline on the development rows

Rule-set coverage is how often the true label is still inside the set. We recompute it for all four threshold variants on the 963 clean development rows.

In [7]:
dev = load_dev(clean=True)
summary = {}
for name, th in VARIANTS.items():
    sets = [run(r.to_dict(), th).compatible for _, r in dev.iterrows()]
    covered = sum(lab in s for lab, s in zip(dev[TARGET], sets))
    summary[name] = {"covered": covered, "coverage %": round(100 * covered / len(dev), 1),
                     "mean set size": round(float(np.mean([len(s) for s in sets])), 2)}
pd.DataFrame(summary).T.astype({"covered": int})

,covered,coverage %,mean set size
primary,784,81.4,1.47
hgb12,634,65.8,1.42
mchc32,929,96.5,1.47
hgb12_mchc32,741,76.9,1.42


## 5. Where the cited rules and the labels disagree

Rows whose values fall between a cited cut-off and the cut-off the labels seem to use.

In [8]:
g_wbc = (dev["WBC"] > 10) & (dev["WBC"] <= 11)
normo_anemia = (dev["HGB"] < 13) & dev["MCV"].between(80, 100)
g_mchc = normo_anemia & (dev["MCHC"] >= 30) & (dev["MCHC"] < 32)
n_wbc, n_mchc = int(g_wbc.sum()), int(g_mchc.sum())
print(f"WBC in (10, 11]: {n_wbc} rows ->", dev.loc[g_wbc, TARGET].value_counts().to_dict())
print(f"normocytic anemia with MCHC in [30, 32): {n_mchc} rows ->", dev.loc[g_mchc, TARGET].value_counts().to_dict())

WBC in (10, 11]: 35 rows -> {'Leukemia': 10, 'Normocytic hypochromic anemia': 8, 'Iron deficiency anemia': 8, 'Leukemia with thrombocytopenia': 6, 'Normocytic normochromic anemia': 3}
normocytic anemia with MCHC in [30, 32): 145 rows -> {'Normocytic hypochromic anemia': 145}


## 6. The frozen label-cascade baseline

This is the probable labeling rule from Phase 1 with one fitted value. It was frozen on 14 Nov and will be scored once on the test set in Phase 6. It is not a clinical rule.

In [9]:
import json
print(json.dumps(json.loads((ROOT / "results" / "phase2" / "cascade_frozen.json").read_text(encoding="utf-8")), indent=2))

{
  "hgb": 13.0,
  "plt": 150.0,
  "wbc": 10.0,
  "mcv_micro": 80.0,
  "mcv_macro": 100.0,
  "mchc_normo": 32.0,
  "mchc_ida_oma": 31.9,
  "note": "Probable labeling rule read off DEVELOPMENT data (Phase 1 audit + one fitted MCHC split). Not a clinical rule. Frozen 14 Nov 2026. Score once on the test set in Phase 6.",
  "fitted_on": "963 clean development rows",
  "fit_rule": "MCHC split maximising balanced accuracy between iron deficiency and other microcytic on development rows labelled with either, grid 28.0-34.0 step 0.1, lowest value at the maximum"
}


## 7. Did the re-run reproduce the committed results?

The re-run report and frozen baseline must be identical to the committed ones, and the coverage numbers and gap counts recomputed above must match them.

In [10]:
assert (P2 / "rules_baseline.txt").read_text(encoding="utf-8") == before, "rules report changed"
assert (P2 / "cascade_frozen.json").read_text(encoding="utf-8") == before_cascade, "frozen cascade changed"
txt = before
saved = [int(x) for x in re.findall(r"rule-set coverage: (\d+) / 963", txt)]
assert saved == [v["covered"] for v in summary.values()], (saved, summary)
assert n_wbc == int(re.search(r"WBC in \(10, 11\].*?: (\d+) rows", txt).group(1))
assert n_mchc == int(re.search(r"MCHC in \[30, 32\).*?: (\d+) rows", txt).group(1))
print("re-run identical to the committed report, recomputed numbers match:", saved, n_wbc, n_mchc)

re-run identical to the committed report, recomputed numbers match: [784, 634, 929, 741] 35 145


## 8. Questions for you (answer without looking)

1. What does forward chaining mean, and what stops it from running forever?
2. Which cited cut-off causes most of the rule-set misses, and how many rows does it cost?
3. Why do the three indices appear in the trace but never change the label set?